# Mechanism transfer: predicting an UNSEEN ligand class

**The question.** We compare **three** receptor representations that differ in what they
know, tested on a chemical class the graph **never saw during training**:
- **raw ESM** — *structure only*: sequence information, sees no binding at all;
- **GNN on ESM** — *structure + function*: our titular graph, ESM node features refined by binding;
- **GNN on one-hot** — *function only*: the same graph fed pure receptor identity (one-hot) instead
  of ESM, so its receptor geometry is learned **entirely from binding**, with zero structural prior.

If knowing structure *and* function beats either alone at predicting an unseen mechanism, the two
signals are complementary; if the one-hot (function-only) graph already matches ESM-GNN, structure
was not needed for this transfer; if it collapses, structure is carrying the generalization.

**Weak (testable) formulation.** Pick a chemically-coherent, mechanistically-meaningful
odorant class C (carboxylic acids -> the Class I "acid receptor" mechanism; thiols -> the
Cu-coordination mechanism). **Remove every molecule of C from training entirely.** Train the
receptor graph on the rest, so its receptor embeddings are built without a single example of
C. Then ask: does the refined receptor geometry predict each receptor's **held-out response
to C** better than raw ESM? If yes, the embedding encodes a *transferable mechanism*, not
memorised statistics — the mechanism "shows itself" on unseen examples.

We test our **titular** pipeline (q99 signed graph) **and** a **full-coverage** variant
(q=0, no quantile), over **5 seeds**, with a GPU attached in-notebook.

*Honest caveat.* A broadly-tuned receptor binds a bit of everything, so some transfer is
generic tuning rather than the specific C-mechanism. Raw ESM (which sees no binding at all)
is the control: any refined-over-ESM gain is binding-derived signal that generalised to the
unseen class. M2OR is large and diverse, so it cannot be fully "surprised" — read the gaps,
not the absolute numbers.


In [ ]:
%matplotlib inline
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr, t as _t

from orbind.regimes import full_full_pairs
from orbind.gnn_extractor import (
    GnnSignedExtractor, _build_universe, _mp_edges, _edge_index_dict, _train_one, MOL, PROT)

DATA   = ROOT / "data"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT:", ROOT, "| device:", DEVICE)

In [ ]:
# ---- configuration -------------------------------------------------------
PROT_NPZ = DATA / "embeddings/proteins/esm1b_650m_mean.npz"
MOL_NPZ  = DATA / "embeddings/molecules/gin_supervised_contextpred_all_m2or.npz"

# graph-config variants: our titular q99 signed graph, and full-coverage (q=0, no quantile)
VARIANTS = {
    "q99 (titular)": dict(q=0.99, criterion="greedy_pair_cover", k_mode="coverage_quantile"),
    "full (q=0)":    dict(q=0.0,  criterion="coverage",          k_mode="coverage_quantile"),
}
SEEDS = [42, 43, 44, 45, 46]
EPOCHS = 900          # titular protocol; lower for a quick dry run

# mechanistically-meaningful odorant classes (SMARTS on the SMILES)
SMARTS = {
    "carboxylic_acid": "[CX3](=O)[OX2H1]",   # Class I acid-receptor mechanism
    "thiol":           "[#16X2H]",           # Cu/Zn-coordination mechanism
    "aldehyde":        "[CX3H1](=O)[#6]",
    "ester":           "[#6][CX3](=O)[OX2H0][#6]",
}
RUN_CLASSES = ["carboxylic_acid"]   # only the flagship is trained; other classes' numbers
                                    # are already recorded, so we do not re-train on them
K_NN = 10                                    # neighbours for the LOO readout

## 1 · Molecules per class (RDKit) and the held-out receptor labels

Each class is a SMARTS substructure. A receptor's **held-out label** is its response to that
class: `1` if it binds at least one class-C molecule (among those it was tested on), else `0`
— computed only from the C sub-matrix, which never enters training.


In [ ]:
pairs = full_full_pairs()
uniq  = pairs.drop_duplicates("inchikey")[["inchikey", "smiles"]]
mols  = {ik: Chem.MolFromSmiles(s) for ik, s in zip(uniq["inchikey"], uniq["smiles"])}

def class_molecules(cname):
    q = Chem.MolFromSmarts(SMARTS[cname])
    return {ik for ik, m in mols.items() if m is not None and m.HasSubstructMatch(q)}

CLASS_IKS = {c: class_molecules(c) for c in RUN_CLASSES}
for c, iks in CLASS_IKS.items():
    sub = pairs[pairs["inchikey"].isin(iks)]
    resp = sub[sub["label"] == 1]["receptor"].nunique()
    print(f"{c:16} molecules={len(iks):3d} | receptors tested-on={sub['receptor'].nunique():4d} "
          f"| responders(>=1)={resp:4d}")

## 1b · See the class: binding table + molecular skeletons

Before any modelling, look at the carboxylic acids with your own eyes. First the binding
table with the acid molecules pulled into a block on the left — if the acid columns light up
a *different* set of receptors than the rest, their response is genuinely atypical. Then the
2D structures: every acid carries the same `-C(=O)-O-H` group (the little "hook"), versus
random training molecules and the chemically *closest* non-acids.


In [ ]:
import matplotlib.pyplot as plt
from rdkit.Chem import Draw, AllChem, DataStructs

CVIZ  = "carboxylic_acid"
smi_of = uniq.set_index("inchikey")["smiles"].to_dict()
acids  = sorted(CLASS_IKS[CVIZ])
others = [ik for ik in uniq["inchikey"] if ik not in CLASS_IKS[CVIZ]]
rng    = np.random.default_rng(0)

# --- binding table: acid block (left) vs an equal random non-acid block -------
samp  = list(rng.choice(others, size=min(len(acids), len(others)), replace=False))
cols  = acids + samp
o_i   = {ik: j for j, ik in enumerate(cols)}
sub   = pairs[pairs["inchikey"].isin(set(cols))]
top_r = sub["receptor"].value_counts().head(70).index.tolist()
r_i   = {r: i for i, r in enumerate(top_r)}
M = np.full((len(top_r), len(cols)), np.nan)
for (r, ik), v in sub[sub["receptor"].isin(set(top_r))].groupby(["receptor", "inchikey"])["label"].max().items():
    M[r_i[r], o_i[ik]] = v
cmap = plt.cm.RdYlGn.copy(); cmap.set_bad("lightgray")
fig, ax = plt.subplots(figsize=(12, 7))
im = ax.imshow(np.ma.masked_invalid(M), aspect="auto", cmap=cmap, vmin=0, vmax=1, interpolation="nearest")
ax.axvline(len(acids) - 0.5, color="black", lw=2.5)
ax.set_xlabel(f"molecules   [ {len(acids)} carboxylic acids | {len(samp)} random others ]")
ax.set_ylabel(f"{len(top_r)} most-measured receptors")
ax.set_title("M2OR binding: green=binds, red=no, grey=untested — acid block is left of the line")
ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, fraction=0.03, label="binds"); fig.tight_layout(); plt.show()

In [ ]:
def fp(ik):
    m = mols.get(ik)
    return AllChem.GetMorganFingerprintAsBitVect(m, 2, 2048) if m is not None else None

def show_grid(iks, title, n=18):
    ms, legs = [], []
    for ik in iks[:n]:
        m = mols.get(ik)
        if m is not None: ms.append(m); legs.append("")
    print(title)
    display(Draw.MolsToGridImage(ms, molsPerRow=6, subImgSize=(160, 120), legends=legs))

fa   = [fp(ik) for ik in acids if fp(ik) is not None]
near = sorted(((ik, max(DataStructs.BulkTanimotoSimilarity(fp(ik), fa))) for ik in others if fp(ik) is not None),
              key=lambda t: -t[1])
show_grid(acids, f"Carboxylic acids — the held-out class ({len(acids)} molecules); note the same -COOH hook")
show_grid([ik for ik, _ in near], "Nearest NON-acid training molecules (highest Tanimoto to the acid group)")
show_grid(list(rng.choice(others, min(18, len(others)), replace=False)), "Random training molecules")

## 2 · Train the graph WITHOUT the class, per variant x seed

For each class we drop all its molecules, build the universe on what remains, and train the
signed graph (titular and full-coverage), five seeds each. `emit="prot"` returns the refined
receptor vector — trained without a single class-C example. Raw ESM is the same for all and
serves as the structural baseline.


In [ ]:
def refined_receptors(variant_kw, seed, drop_iks, feat="esm"):
    """Train the signed graph on all pairs whose molecule is NOT in drop_iks;
    return {receptor: refined vector} and {receptor: raw ESM vector}.

    feat='esm'    -> ESM node features (structure + function).
    feat='onehot' -> replace protein features with a one-hot identity over the SAME receptor
                     set ESM covers, so the receptor geometry is learned from binding alone
                     (function only, no structural prior). Same universe, same edges."""
    ext = GnnSignedExtractor(name="cls", protein_path=str(PROT_NPZ), molecule_path=str(MOL_NPZ),
                             emit="prot", n_models=1, hidden=256, edge_threshold=0.0,
                             task="classification", epochs=EPOCHS, **variant_kw)
    if feat == "onehot":
        recs = sorted(ext._proteins)                       # exactly the receptors ESM covers
        eye  = np.eye(len(recs), dtype=np.float32)
        ext._proteins = {r: eye[i] for i, r in enumerate(recs)}
    keep = ~pairs["inchikey"].isin(drop_iks)
    idx  = np.where(keep.to_numpy())[0]
    idx  = idx[ext.covered(pairs, idx)]
    mol_to_i, prot_to_i, x_mol, x_prot = _build_universe(pairs, idx, ext._proteins, ext._molecules)
    pos, neg = _mp_edges(pairs, idx, mol_to_i, prot_to_i, ext.q, ext.criterion,
                         ext.edge_threshold, ext.k_mode, ext.task)
    pe, ne = _edge_index_dict(pos, neg)
    sub = pairs.iloc[idx]
    mi = sub["inchikey"].map(mol_to_i).to_numpy()
    pi = sub["receptor"].map(prot_to_i).to_numpy()
    y  = sub["label"].to_numpy(dtype=np.float32)
    torch.manual_seed(seed)
    _, z_prot, _ = _train_one(ext._build_model, x_mol, x_prot, pe, ne, mi, pi, y,
                              ext._hp(seed), DEVICE, checkpoint_path=None)
    refined = {r: z_prot[i] for r, i in prot_to_i.items()}
    rawesm  = {r: x_prot[i].numpy() for r, i in prot_to_i.items()}
    return refined, rawesm

In [ ]:
def heldout_label(cname, receptors):
    """Binary C-response + the per-C-molecule profile, for the given receptors."""
    iks = CLASS_IKS[cname]
    sub = pairs[pairs["inchikey"].isin(iks)]
    cmols = sorted(iks)
    m_idx = {m: j for j, m in enumerate(cmols)}
    prof  = {}
    label = {}
    for r, g in sub.groupby("receptor"):
        v = np.full(len(cmols), np.nan)
        agg = g.groupby("inchikey")["label"].max()
        for ik, val in agg.items():
            v[m_idx[ik]] = val
        prof[r]  = v
        label[r] = int(np.nanmax(v) >= 1) if np.isfinite(v).any() else None
    recs = [r for r in receptors if label.get(r) is not None]
    y    = np.array([label[r] for r in recs])
    Pc   = np.stack([np.nan_to_num(prof[r], nan=0.0) for r in recs])  # signed-ish profile (0/1)
    return recs, y, Pc

def emb_sim(X):
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    return Xn @ Xn.T

def loo_knn_auroc(X, y, k=K_NN):
    S = emb_sim(X).copy(); np.fill_diagonal(S, -np.inf)
    nn = np.argpartition(-S, k, axis=1)[:, :k]
    score = np.array([y[row].mean() for row in nn])
    return roc_auc_score(y, score) if 0 < y.sum() < len(y) else np.nan

def profile_spearman(X, Pc):
    Fc = Pc - Pc.mean(1, keepdims=True)
    Fn = Fc / (np.linalg.norm(Fc, axis=1, keepdims=True) + 1e-9)
    F  = Fn @ Fn.T
    iu = np.triu_indices(len(X), 1)
    return spearmanr(F[iu], emb_sim(X)[iu]).correlation

## 2b · The picture: actual acid responses vs all three models

Train the **full-coverage (q=0)** graph with every acid removed — once with ESM node features,
once with one-hot identity — then reconstruct each receptor's acid responses from its neighbours
in each embedding (leave-one-out kNN over the held-out acid columns). Four panels, same
green/red palette: **actual** binding, then the three models — **GNN+ESM** (both),
**GNN one-hot** (function only), **ESM** (structure only). Whichever panels track the actual band
carry the acid mechanism; a washed-out panel does not. Comparing the two graph panels isolates
what the structural prior adds on top of pure binding.

**Only measured cells are shown.** M2OR is sparse — most receptor x acid pairs were never assayed
— so we mask every model panel to the *same* support as `actual` (grey = untested, identical in
all four). That way each coloured model cell has a ground-truth counterpart to compare against,
and the panels share one shape; we are asking each embedding exactly about the acid responses
M2OR actually recorded, nothing extrapolated. (The insect notebooks need no such mask: their
matrices are complete.) Actual is binary (bind / no-bind); model cells are graded neighbour
averages, so read a model panel as *predicted binding propensity* on the measured cells.


In [ ]:
import warnings; warnings.simplefilter("ignore", category=RuntimeWarning)

ref_v, raw_v = refined_receptors(VARIANTS["full (q=0)"], 42, CLASS_IKS[CVIZ])              # GNN(ESM)
fun_v, _     = refined_receptors(VARIANTS["full (q=0)"], 42, CLASS_IKS[CVIZ], feat="onehot")  # GNN(one-hot)
order = [r for r in ref_v if r in fun_v]; r_i = {r: i for i, r in enumerate(order)}
acid_cols = sorted(CLASS_IKS[CVIZ]); a_i = {a: j for j, a in enumerate(acid_cols)}
A = np.full((len(order), len(acid_cols)), np.nan)                                # actual acid responses
asub = pairs[pairs["inchikey"].isin(CLASS_IKS[CVIZ]) & pairs["receptor"].isin(set(order))]
for (r, ik), v in asub.groupby(["receptor", "inchikey"])["label"].max().items():
    A[r_i[r], a_i[ik]] = v
Xr = np.stack([ref_v[r] for r in order]); Xe = np.stack([raw_v[r] for r in order])
Xf = np.stack([fun_v[r] for r in order])

def reconstruct(X):
    S = emb_sim(X).copy(); np.fill_diagonal(S, -np.inf)
    nn = np.argpartition(-S, K_NN, axis=1)[:, :K_NN]
    return np.stack([np.nanmean(A[nn[i]], axis=0) for i in range(len(order))])
P_our, P_fun, P_esm = reconstruct(Xr), reconstruct(Xf), reconstruct(Xe)

# M2OR is SPARSE: only score the receptor x acid cells we actually measured, so every coloured
# model cell has a ground-truth counterpart and all panels share the same grey (untested) mask.
mask = np.isfinite(A)
P_our = np.where(mask, P_our, np.nan)
P_fun = np.where(mask, P_fun, np.nan)
P_esm = np.where(mask, P_esm, np.nan)

sel = np.where(np.isfinite(A).any(1))[0]                       # receptors tested on >=1 acid
sel = sel[np.argsort(-np.nanmean(A[sel], axis=1))][:70]        # sorted by acid hit-rate
cmap = plt.cm.RdYlGn.copy(); cmap.set_bad("lightgray")
panels = [(A, "actual"), (P_our, "GNN+ESM (both)"),
          (P_fun, "GNN one-hot (function only)"), (P_esm, "ESM (structure only)")]
fig, axes = plt.subplots(1, 4, figsize=(19, 7), sharey=True)
for ax, (Mx, ttl) in zip(axes, panels):
    ax.imshow(np.ma.masked_invalid(Mx[sel]), aspect="auto", cmap=cmap, vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(ttl, fontsize=10); ax.set_xlabel(f"{len(acid_cols)} acids"); ax.set_xticks([]); ax.set_yticks([])
axes[0].set_ylabel(f"{len(sel)} acid-tested receptors (sorted by acid hit-rate)")
fig.suptitle("M2OR carboxylic acids — actual vs three models (green=binds, grey=untested); acids never seen by any graph")
fig.tight_layout(); plt.show()

In [ ]:
rows = []
for cname in RUN_CLASSES:
    drop = CLASS_IKS[cname]
    # raw ESM baseline is variant/seed-independent: take it from any trained universe once
    base_ref, base_raw = refined_receptors(VARIANTS[next(iter(VARIANTS))], SEEDS[0], drop)
    recs, y, Pc = heldout_label(cname, list(base_raw))
    Xesm = np.stack([base_raw[r] for r in recs])
    rows.append(dict(cls=cname, embedding="raw ESM", feat="-", variant="-", seed="-",
                     auroc=loo_knn_auroc(Xesm, y), spear=profile_spearman(Xesm, Pc)))
    print(f"[{cname}] {len(recs)} receptors, {int(y.sum())} responders | "
          f"raw ESM AUROC {rows[-1]['auroc']:.3f}")
    for feat, tag in [("esm", "GNN+ESM"), ("onehot", "GNN one-hot")]:
        for vname, vkw in VARIANTS.items():
            for seed in SEEDS:
                ref, _ = refined_receptors(vkw, seed, drop, feat=feat)
                Xr = np.stack([ref[r] for r in recs])
                rows.append(dict(cls=cname, embedding=f"{tag} {vname}", feat=feat, variant=vname,
                                 seed=seed, auroc=loo_knn_auroc(Xr, y), spear=profile_spearman(Xr, Pc)))
            au = [r["auroc"] for r in rows if r["cls"] == cname and r["feat"] == feat and r["variant"] == vname]
            print(f"    {tag:12} {vname:14} AUROC {np.nanmean(au):.3f} +/- {np.nanstd(au):.3f}")
res = pd.DataFrame(rows)

## 3 · Who encodes the mechanism? structure vs function vs both

AUROC of predicting the held-out class response from the receptor geometry (leave-one-out
kNN), for all three models: **raw ESM** (structure only), **GNN+ESM** (both), **GNN one-hot**
(function only). Each graph appears at both variants (titular q99 and full-coverage q0). A bar
above the ESM line means that model transferred a mechanism it never trained on; the gap
between GNN+ESM and GNN one-hot is what the structural prior adds on top of pure binding.


In [ ]:
def ci(x):
    x = np.asarray([v for v in x if np.isfinite(v)], float)
    if len(x) < 2: return (np.nanmean(x) if len(x) else np.nan, np.nan)
    return x.mean(), _t.ppf(.975, len(x) - 1) * x.std(ddof=1) / len(x) ** .5

# the models to report, in order: structure-only, then the two graphs x two variants
MODELS = [("raw ESM", None, None)]
for tag, feat in [("GNN+ESM", "esm"), ("GNN one-hot", "onehot")]:
    for vname in VARIANTS:
        MODELS.append((f"{tag} {vname}", feat, vname))

summary = []
for cname in RUN_CLASSES:
    row = {"class": cname}
    for label, feat, vname in MODELS:
        if feat is None:
            row[label] = res[(res.cls == cname) & (res.embedding == "raw ESM")]["auroc"].iloc[0]
            row[f"{label} _ci"] = 0.0
        else:
            m, h = ci(res[(res.cls == cname) & (res.feat == feat) & (res.variant == vname)]["auroc"])
            row[label] = m; row[f"{label} _ci"] = h
    summary.append(row)
summ = pd.DataFrame(summary)
display(summ[[c for c in summ.columns if not c.endswith("_ci")]]
        .style.format({c: "{:.3f}" for c in summ.columns if c not in ("class",) and not c.endswith("_ci")})
        .hide(axis="index"))

# tuning baseline: how well does a receptor's OVERALL non-C hit-rate alone predict its
# class response? Any AUROC near this is generic tuning, not the class mechanism.
tune_auroc = {}
for cname in RUN_CLASSES:
    hit = pairs[~pairs["inchikey"].isin(CLASS_IKS[cname])].groupby("receptor")["label"].mean()
    recs_c, y_c, _ = heldout_label(cname, list(hit.index))
    hr = hit.loc[recs_c].to_numpy()
    tune_auroc[cname] = roc_auc_score(y_c, hr) if 0 < y_c.sum() < len(y_c) else np.nan
    print(f"{cname:16} tuning-only baseline AUROC = {tune_auroc[cname]:.3f}")

fig, ax = plt.subplots(figsize=(11, 4.5))
labels = [m[0] for m in MODELS]
palette = {"raw ESM": "#888888",
           "GNN+ESM q99 (titular)": "#2f5c9e", "GNN+ESM full (q=0)": "#7aa0d4",
           "GNN one-hot q99 (titular)": "#b5651d", "GNN one-hot full (q=0)": "#e8a15c"}
x = np.arange(len(RUN_CLASSES)); w = 0.8 / len(labels)
for j, lab in enumerate(labels):
    vals = [summ.loc[summ["class"] == c, lab].iloc[0] for c in RUN_CLASSES]
    errs = [summ.loc[summ["class"] == c, f"{lab} _ci"].iloc[0] for c in RUN_CLASSES]
    ax.bar(x + (j - (len(labels)-1)/2) * w, vals, w, yerr=errs, capsize=2,
           label=lab, color=palette.get(lab, "#999999"))
for xi, c in enumerate(RUN_CLASSES):
    ax.plot([xi - 0.45, xi + 0.45], [tune_auroc[c]] * 2, ls="--", c="crimson", lw=1.3,
            label="tuning-only baseline" if xi == 0 else None)
ax.axhline(0.5, ls=":", c="gray")
ax.set_xticks(x); ax.set_xticklabels(RUN_CLASSES)
ax.set_ylabel(f"held-out class AUROC (LOO {K_NN}-NN)")
ax.set_title("Predicting an UNSEEN ligand class: structure-only (ESM) vs both (GNN+ESM) vs function-only (GNN one-hot)")
ax.legend(frameon=False, fontsize=8, ncol=2); ax.set_ylim(0.4, 1.0); fig.tight_layout(); plt.show()

**Reading it.** If the graph bars clear the raw-ESM baseline on a class the graph never
trained on, that receptor embedding carries a **transferable binding mechanism**, not just
memorised co-occurrence — the strongest sense in which our receptor side beats ESM. The
**three-way** contrast is the point: **ESM** knows only structure, **GNN one-hot** knows only
function (identity + binding, no sequence), **GNN+ESM** knows both. If both-beats-either, the
signals are complementary; if one-hot already matches ESM-GNN, structure was not needed for this
transfer; if one-hot collapses toward chance, structure is carrying the generalization. Comparing
the **titular q99** and **full-coverage q0** variants shows whether the hub-molecule restriction
helps or hurts.

**Two caveats are drawn on the plot.** M2OR is sparse, so *which* receptor x molecule pairs were
assayed is not random — a confound this binary readout cannot fully remove; and the **red dashed
tuning-only baseline** is the AUROC obtained from each receptor's overall non-C hit-rate alone
(generic responsiveness). Any bar near that line is tuning, not the class mechanism — only the
margin *above* it is a specific-mechanism signal. The insect complete-matrix notebook
(`mechanism_holdout_insects.ipynb`) removes both confounds by construction (uniform assay
pattern) and by residualising the target against tuning; treat this M2OR result as the
illustrative version and that one as the clean test.
